- The goal of this examination is to find out how, imperfections (or plain error) in the underlying models are reflected in the resulting data.
- The question is whether DMPE is potentially less sensitive to model errors compared to other approaches where the loss function is directly based on the model prediction (e.g. model uncertainty, ensembles, etc.)
- TODOs:
    - create imperfect models
    - run DMPE
    - inspect the relationship between model error and JSD 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_debug_nans", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence
import jax_dataclasses as jdc

from dmpe.data_management import DataPaths
from dmpe.evaluation.plotting_utils import plot_sequence, plot_feature_combinations
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results
from dmpe.models.models import NeuralEulerODECartpole
from dmpe.models.model_utils import simulate_ahead_with_env

In [ ]:
from dmpe.data_management import DataPaths
from dmpe.utils.signals import aprbs
from dmpe.utils.density_estimation import select_bandwidth, get_uniform_target_distribution
from dmpe.algorithms.algorithms import excite_with_dmpe
from dmpe.models.models import NeuralEulerODEPendulum, NeuralEulerODE, NeuralEulerODECartpole
from dmpe.models.model_utils import save_model
from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

from dmpe.algorithms.algorithms import excite_with_dmpe
from dmpe.algorithms.algorithm_utils import default_dmpe_parameterization

import exciting_environments as excenvs
from dmpe.algorithms.algorithm_utils import (
    consult_exciter,
    interact_and_observe,
    default_dmpe_parameterization,
)
from dmpe.excitation.excitation_utils import loss_function, Exciter
from dmpe.models.model_training import ModelTrainer
from dmpe.utils.density_estimation import (
    DensityEstimate,
    build_grid,
)
from dmpe.utils.metrics import JSDLoss

from dmpe.algorithms.algorithms import excite_and_fit
from dmpe.evaluation.imperfect_pm_dmpe import excite_with_imperfect_model, adapt_static_params

from dmpe.evaluation.model_evaluation import ModelEvaluator, EnvWrapper
from dmpe.evaluation.data_evaluation import DataEvaluator
from dmpe.evaluation.utils import default_constraint_function

In [ ]:
from exciting_environments import CartPole

In [ ]:
env, penalty_function, featurize, env_params = setup_pendulum_env()
# env, penalty_function, featurize, env_params = setup_cart_pole_env()

In [ ]:
obs_dim = env.reset(env.env_properties)[0].shape[-1]
act_dim = env.action_dim

wrapped_env = EnvWrapper(env, featurize)

points_per_dim = 20

model_evaluator = ModelEvaluator(
    constraint_function=default_constraint_function,
    gt_model=wrapped_env,
    obs_dim=obs_dim,
    act_dim=act_dim,
    validation_points_per_dim=points_per_dim,
    tau=env.tau,
)

data_evaluator = DataEvaluator(
    constraint_function=default_constraint_function,
    data_dim=obs_dim + act_dim,
    points_per_dim=points_per_dim,
)

In [ ]:
sys_name = "fluid_tank"

In [ ]:
if sys_name == "fluid_tank":
    env, penalty_function, featurize, env_params = setup_fluid_tank_env()
    parameter_ranges = {}
    parameter_ranges = dict(
        base_area=jnp.pi * jnp.arange(0.6, 1.41, 0.2),
        orifice_area=jnp.pi * 0.1**2 * jnp.arange(0.6, 1.41, 0.2),
    )
    parameter_combinations = jnp.meshgrid(*parameter_ranges.values(), indexing="ij")
    n_parameters = len(parameter_combinations)
    parameter_combinations = jnp.stack([_x for _x in parameter_combinations], axis=-1)
    parameter_combinations = parameter_combinations.reshape(-1, n_parameters)

elif sys_name == "pendulum":
    env, penalty_function, featurize, env_params = setup_pendulum_env()
    parameter_ranges = dict(
        m=jnp.arange(0.6, 1.41, 0.2),
        l=jnp.arange(0.6, 1.41, 0.2),
    )
    parameter_combinations = jnp.meshgrid(*parameter_ranges.values(), indexing="ij")
    n_parameters = len(parameter_combinations)
    parameter_combinations = jnp.stack([_x for _x in parameter_combinations], axis=-1)
    parameter_combinations = parameter_combinations.reshape(-1, n_parameters)

elif sys_name == "cart_pole":
    env, penalty_function, featurize, env_params = setup_cart_pole_env()
    parameter_ranges = dict(
        m_c=jnp.arange(0.6, 1.41, 0.2),
        m_p=jnp.arange(0.1, 1.11, 0.2),
        l=jnp.arange(0.1, 1.11, 0.2),
    )
    parameter_combinations = jnp.meshgrid(*parameter_ranges.values(), indexing="ij")
    n_parameters = len(parameter_combinations)
    parameter_combinations = jnp.stack([_x for _x in parameter_combinations], axis=-1)
    parameter_combinations = parameter_combinations.reshape(-1, n_parameters)

In [ ]:
parameter_combinations

In [ ]:
default_params = env.env_properties.static_params

all_jsd_values = []
all_me_values = []

for parameters in tqdm(parameter_combinations):
    print(parameters)

    if sys_name == "fluid_tank":
        A, A_o = tuple(parameters)
        new_static_params=env.StaticParams(
            base_area=A.item(),
            orifice_area=A_o.item(),
            c_d=default_params.c_d,
            g=default_params.g,
        )
    elif sys_name == "pendulum":
        m, l = tuple(parameters)
        new_static_params=env.StaticParams(
            l=l.item(),
            m=m.item(),
            g=default_params.g,
        )

    elif sys_name == "cart_pole":
        m_c, m_p, l = tuple(parameters)
        new_static_params = env.StaticParams(
            l=l.item(),
            m_p=m_p.item(),
            m_c=m_c.item(),
            g=default_params.g,
            mu_p=default_params.mu_p,
            mu_c=default_params.mu_c,
        )
    
    jsd_values = []
    me_values = []
    
    for seed in [77845]:#, 12341, 15152, 37431]:# 155, 166, 515]:
        observations, actions, model, exp_params, debug_out = excite_with_imperfect_model(
            seed=seed,
            n_time_steps=10,
            env=env,
            penalty_function=penalty_function,
            env_params=env_params,
            static_params=new_static_params,
        )
        
        # fig = plot_sequence(observations, actions, env.tau, env.obs_description, env.action_description)
        # plt.show()
        
        # plot_feature_combinations(
        #     data=jnp.concatenate([observations[:-1], actions], axis=-1),
        #     labels=[*env.obs_description, *env.action_description],
        #     mode="contourf",
        #     bandwidth=0.08,
        # )
        # plt.show()
        
        _, me_value = model_evaluator.default_metrics["pred_comp"](wrapped_env, EnvWrapper(model, featurize))
        print(f"model_error: {me_value}")
        jsd_value = data_evaluator.default_metrics["jsd"](jnp.concatenate([observations[:-1], actions], axis=-1))
        print(f"jsd_value: {jsd_value}")

        jsd_values.append(jsd_value)
        me_values.append(me_value)
    all_jsd_values.append(jsd_values)
    all_me_values.append(me_values)

In [ ]:
new_static_params = 

In [ ]:
import jax_dataclasses as jdc

In [ ]:
jdc.asdict(new_static_params)

In [ ]:
new_static_params.asdict

In [ ]:
average_jsd = [np.mean(jsd_values) for jsd_values in all_jsd_values]
average_me = [np.mean(me_values) for me_values in all_me_values]

In [ ]:
average_jsd.pop(np.argmin(average_me))
average_me.pop(np.argmin(average_me))

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(8,8))
ax.plot(average_jsd, average_me, ".")

ax.set_yscale("log")

In [ ]:
# filename = DataPaths().imperfect_pm_dmpe_experiments / "pendulum_imperfect_pm_dmpe_experiments.json"


# data = dict(
#     all_jsd_values=all_jsd_values,
#     all_me_values=all_me_values
# )

# with open(filename, "w") as f:
#     json.dump(data, f)

## plotting stuff:

In [ ]:
key = jax.random.PRNGKey(1)
key, action_key = jax.random.split(key, 2)

init_obs, init_state = env.reset(env.env_properties)
init_state = env.generate_state_from_observation(init_obs, env.env_properties)

actions = aprbs(100, 1, 10, 20, action_key)[0]
observations, last_state = simulate_ahead_with_env(
    env, init_obs=init_obs, init_state=init_state, actions=actions
)

fig, axs = plt.subplots(1,2,figsize=(12,6))

axs[0].plot(observations[:, 0], "b", linestyle="dashdot")
axs[1].plot(observations[:, 1], "b", linestyle="dashdot")
# axs[2].plot(observations[:, 2], "b", linestyle="dashdot")
# axs[3].plot(observations[:, 3], "b", linestyle="dashdot")

l = jnp.array([default_params.l])
# m = jnp.array([default_params.m_p])

for parameters in parameter_combinations:
    if sys_name == "fluid_tank":
        raise NotImplementedError()
    elif sys_name == "pendulum":
        m, l = tuple(parameters)
        new_static_params=env.StaticParams(
            l=l.item(),
            m=m.item(),
            g=default_params.g,
        )

    elif sys_name == "cart_pole":
        m_c, m_p, l = tuple(parameters)
        new_static_params = env.StaticParams(
            l=l.item(),
            m_p=m_p.item(),
            m_c=m_c.item(),
            g=default_params.g,
            mu_p=default_params.mu_p,
            mu_c=default_params.mu_c,
        )
    model = adapt_static_params(
        env, 
        new_static_params=new_static_params,
    )
    
    observations, last_state = simulate_ahead_with_env(
        model, init_obs=init_obs, init_state=init_state, actions=actions
    )
    
    axs[0].plot(observations[:, 0], "r--")
    axs[1].plot(observations[:, 1], "r--")
    # axs[2].plot(observations[:, 2], "r--")
    # axs[3].plot(observations[:, 3], "r--")

for ax in axs:
    ax.grid(True)
plt.show()

- integrate more seamlessly into the existing algorithm
- setup for all systems
- actually compare the jsds and model accuracy through model evaluator and data evaluator objects


In [ ]:
obs_dim = env.reset(env.env_properties)[0].shape[-1]
act_dim = env.action_dim

wrapped_env = EnvWrapper(env, featurize)

points_per_dim = 20

model_evaluator = ModelEvaluator(
    constraint_function=default_constraint_function,
    gt_model=wrapped_env,
    obs_dim=obs_dim,
    act_dim=act_dim,
    validation_points_per_dim=points_per_dim,
    tau=env.tau,
)

In [ ]:
g = 9.81
m = 1
for l in jnp.arange(0.1, 1.9, 0.1):
    model = adapt_static_params(env, env.StaticParams(g=g, l=l.item(), m=m))
    loss_map, value = model_evaluator.default_metrics["pred_comp"](wrapped_env, EnvWrapper(model, featurize))
    print(f"l: {l}, value:{value}")

In [ ]:
from dmpe.evaluation.model_evaluation import visualize_model_prediction_performance

In [ ]:
g = 9.81
m = 0.9
l = 1

model = adapt_static_params(env, env.StaticParams(g=g, l=l, m=m))
visualize_model_prediction_performance(
    EnvWrapper(model, featurize),
    model_evaluator,
    labels=[*env.obs_description, *env.action_description],
)
plt.show()